# Música em Contexto: Music Brainz + Spotify

## Camada Gold

Nessa camada eu analiso **contexto_silver**, trabalhando valores nulos e comparando com as outras tabelas para criar a **contexto_gold**

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA gold;

In [0]:
from pyspark.sql import functions as F

Importação de **contexto_silver** como **df_contexto_silver**

In [0]:
df_contexto_silver = spark.read.table("mvp.silver.contexto_silver")

display(df_contexto_silver.limit(100))

### Tratamento de Valores Nulos

In [0]:
df_contexto_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_contexto_silver.columns
]).show()

A única coluna na qual existem valores nulos é a **contry**. São 1190983 valores nulos no total.

Calculando a proporção de nulos na tabela:

In [0]:
df_contexto_silver.select(
    F.count("*").alias("total_linhas"),
    F.count(F.when(F.col("country").isNull(), 1)).alias("nulos_country")
).withColumn(
    "percentual_nulos",
    F.col("nulos_country") / F.col("total_linhas") * 100
).show()

Eu tenho 11% de valores nulos para **country**. Não é possível recuperar esses valores de outras forma, vou ter que cortar essas linhas do meu estudo.

In [0]:
df_contexto_gold = (
    df_contexto_silver
    .filter(F.col("country").isNotNull())
)

As instâncias que continham valores nulos em **country** foram cortadas do estudo, dando origem à primeira versão da dataframe **df_contexto_gold**

Não há mais valores nulos:

In [0]:
df_contexto_gold.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_contexto_gold.columns
]).show()

### Limpeza de Colunas Desnecessárias ou Redundantes
Aqui eu retiro as colunas que não tem utilidade para o estudo ou que são redundantes.

In [0]:
SEED = 42

In [0]:
display(
    df_contexto_gold
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(100)
)

In [0]:
df_contexto_gold.columns

In [0]:
df_contexto_gold = df_contexto_gold.drop(
    "artist_mbids",
    "artist_name",
    "album_name",
    "track_name",
    "cl_normal",
    "cl_normal_spotify",
    "name",
    "key",
    "mode",
    "energy_danceability_score"
)

Justificativa das exclusões:

| Coluna                      | Motivo                                                                                |
| --------------------------- | ------------------------------------------------------------------------------------- |
| `artist_mbids`              | Identificador técnico do MusicBrainz; útil para o join, mas não para a análise final. |
| `artist_name`               | Redundante com `artist_credit_name`.                                                  |
| `album_name`                | Redundante com `release_name`.                                                        |
| `track_name`                | Redundante com `recording_name`.                                                      |
| `cl_normal`                 | Coluna intermediária de matching.                                                     |
| `cl_normal_spotify`         | Coluna intermediária de matching.                                                     |
| `name`                      | Nome do artista vindo do MusicBrainz; redundante com `artist_name`.                   |
| `key`                       | Variável que se mostrou pouco útil em estudos anteriores.                             |
| `mode`                      | Variável que se mostrou pouco útil em estudos anteriores.                             |
| `energy_danceability_score` | Variável derivada.                                                                    |


### Conhecendo a Tabela Final

In [0]:
display(
    df_contexto_gold
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(100)
)

In [0]:
df_contexto_gold_shape = (df_contexto_gold.count(), len(df_contexto_gold.columns))

print(df_contexto_gold_shape)

A tabela final tem mais de 9 milhões de instâncias e 20 atributos.

Salvando **df_contexto_gold** na camada **Gold** como **contexto_gold**

In [0]:
df_contexto_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.gold.contexto_gold")

### Gerando o Catálogo de Dados

In [0]:
descricoes = {
    "artist_name": "Nome do artista.",
    "track_name": "Nome da faixa musical.",
    "album_name": "Nome do álbum.",
    "album_release_date": "Data de lançamento do álbum.",
    "duration_ms": "Duração da faixa em milissegundos.",
    "explicit": "Indica se a faixa possui conteúdo explícito.",
    "track_popularity": "Indicador de popularidade da faixa.",
    "artist_popularity": "Indicador de popularidade do artista.",
    "artist_followers": "Número de seguidores do artista.",
    "tempo": "Andamento estimado da faixa, em batidas por minuto (BPM).",
    "danceability": "Medida de adequação da faixa para dança.",
    "energy": "Medida perceptual de intensidade e atividade da faixa.",
    "loudness": "Volume médio da faixa, medido em decibéis (dB).",
    "speechiness": "Medida da presença de elementos de fala na faixa.",
    "acousticness": "Medida de confiança de que a faixa é acústica.",
    "instrumentalness": "Medida de probabilidade de a faixa não conter vocais.",
    "liveness": "Medida da probabilidade de a faixa ter sido gravada ao vivo.",
    "valence": "Medida associada à positividade e ao humor musical da faixa.",
    "country": "País ou território associado ao artista segundo os dados do MusicBrainz.",
    "decade": "Década correspondente ao ano de lançamento do álbum."
}

In [0]:
linhas = []

linhas.append("# Catálogo de Dados — contexto_gold\n")
linhas.append("| Coluna | Tipo | Nulo? | Descrição |")
linhas.append("|---|---|---|---|")

for linha in df_catalogo.collect():
    descricao = descricoes.get(linha["Coluna"], "")
    
    linhas.append(
        f"| {linha['Coluna']} | {linha['Tipo']} | "
        f"{linha['Nulo?']} | {descricao} |"
    )

markdown = "\n".join(linhas)

print(markdown)